# Tutorial — EnergyScope Pathway Model

This notebook shows how to run the EnergyScope transition-pathway model programmatically using `run_pathway()` from `shared.utils`, access the results, and generate plots.

Make sure you are running this notebook from the repository root (or that the repo root is on your Python path), and that the `shared` package is installed:
```bash
pip install -e .
```

## 1. Import

In [7]:
from shared.utils import run_pathway

## 2. Basic run

`run_pathway()` runs the full rolling-horizon optimisation and returns the results as a plain Python dictionary of pandas DataFrames — no file I/O required unless you explicitly ask for it.

In [5]:
results = run_pathway('test_scenario_reasliste')

[ERROR] 
	/Users/Paolo/Documents/PdM_code/EnergyScope-Quebec/projects/pathway/model/PES_scenarios.mod
	line 99 offset 4749
	syntax error
	context:   >>> */ <<< 


/Users/Paolo/Documents/PdM_code/EnergyScope-Quebec/projects/pathway/model/PES_scenarios.mod
line 99 offset 4749
syntax error
context:   >>> */ <<< 

For support/feedback go to https://discuss.ampl.com or e-mail <support@ampl.com>



AMPLException: /Users/Paolo/Documents/PdM_code/EnergyScope-Quebec/projects/pathway/model/PES_scenarios.mod
line 99 offset 4749
syntax error
context:   >>> */ <<< 

For support/feedback go to https://discuss.ampl.com or e-mail <support@ampl.com>


## 3. Exploring results

The returned dictionary contains ~30 named DataFrames. Here are the most commonly used ones:

In [ ]:
# All available result keys
print([k for k in results if results[k] is not None])

In [ ]:
# Installed capacity per technology and year [GW]
results['F_Mult'].loc['YEAR_2030']

In [ ]:
# Annual GHG emissions per year [kt CO2-eq.]
results['TotalGwp']

In [ ]:
# Cumulative transition cost per window-end year [M$CAD]
results['Transition_cost']

In [ ]:
# Energy balance per layer (rows = (year, tech/resource), columns = layers) [GWh]
results['Year_balance'].head(10)

## 4. Plotting

Pass the results dict directly to `plot_results.run()` to generate all HTML charts, or use `plot=True` in `run_pathway()` to do it automatically after the optimisation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
import plot_results

plot_results.run(results, case_study='my_first_run')

Or equivalently, let `run_pathway` handle plotting in one shot:

In [8]:
results = run_pathway('test_scenario_realiste', plot=True)

Unexpected end of file while reading AMPL output.
Usually this is caused by the termination of the underlying AMPL interpreter.


RuntimeError: Unexpected end of file while reading AMPL output.
Usually this is caused by the termination of the underlying AMPL interpreter.

## 5. Saving results to disk

Set `save_pkl=True` to write the results to `projects/pathway/out/<case_study>/_Results.pkl`. You can reload them later with `load_results()` from `plot_results`, or skip a run that already exists with `skip_if_exists=True`.

In [ ]:
results = run_pathway(
    'my_first_run',
    save_pkl=True,
    description='Baseline run with default settings',
    skip_if_exists=True,   # skip if already computed
)

## 6. GWP budget

Set `gwp_budget=True` to activate the whole-transition cumulative CO₂ budget constraint using the built-in default value (1 224 935 kt CO₂-eq.), or pass a custom float.

> **Note:** this requires the `gwp_limit_transition` constraint to be active in the model files.

In [ ]:
# Using the default budget value
results_budget = run_pathway('budget_default', gwp_budget=True)

# Using custom value for the whole-transition GWP budget (in kt CO2-eq.)
results_budget = run_pathway('budget_custom', gwp_budget=100000)

# Without CO2 Budget (still emissions constraints in years 2035-2050)
results_tight = run_pathway('budget_tight') #default case

## 7. Adding extra model files

Use `extra_files` to inject additional `.mod` or `.dat` files into the model — for new constraints, parameter overrides, or scenario definitions. Files are loaded after the standard data files but before `fix.mod`, so they can reference all sets and parameters.

In [ ]:
results_scenario = run_pathway(
    'my_scenario',
    extra_files=[
        '../model/test_file.dat',   # e.g. new technology limits
    ],
    plot=True,
)

## 8. Myopic (rolling-horizon) optimisation

`N_year_opti` sets the width of each optimisation window in years (default `30`, i.e. one window covering the whole horizon, giving full look-ahead / "perfect foresight"). Set it to `5` for fully **myopic** mode, where each 5-year phase is solved on its own, blind to anything beyond it.

`N_year_overlap` sets how many years consecutive windows share. For example, `N_year_opti=10, N_year_overlap=5` solves 10-year windows that each re-use the last 5 years of the previous window, giving a middle ground between full foresight and fully myopic.

In [ ]:
# Myopic mode example: each 10-year phase solved independently with no 5 years overlap.
results_myopic = run_pathway(
    'my_myopic_run',
    N_year_opti=10,
    N_year_overlap=5,
    plot=True,
)